In [ ]:
# Builtin retry for flaky external endpoints (httpbingo.org) - retries transient network errors + 5xx/429.
import time as _t

def _retry(_fn, _tries=5, _delay=1.0):
    for _i in range(_tries):
        try:
            _r = _fn()
            if getattr(_r, 'status_code', 200) in (429, 500, 502, 503, 504) and _i < _tries - 1:
                _t.sleep(_delay * (_i + 1)); continue
            return _r
        except Exception:
            if _i == _tries - 1: raise
            _t.sleep(_delay * (_i + 1))

In [ ]:
# Example 1: CachedSession with in-memory backend, verify cache hit/miss semantics.
from requests_cache import CachedSession

session = CachedSession(backend='memory', expire_after=300)
url = 'https://httpbingo.org/get'

r1 = _retry(lambda: session.get(url))
r2 = _retry(lambda: session.get(url))

print(f"r1.from_cache = {r1.from_cache}")
print(f"r2.from_cache = {r2.from_cache}")
print(f"r1.status_code = {r1.status_code}")

assert r1.from_cache is False
assert r2.from_cache is True
assert r1.status_code == 200


In [ ]:
# Example 2: SQLite backend with a temp directory; verify persistence + cache.contains().
import tempfile
from pathlib import Path

from requests_cache import CachedSession

with tempfile.TemporaryDirectory() as tmpdir:
    db_path = Path(tmpdir) / 'http_cache.sqlite'
    session = CachedSession(str(db_path), backend='sqlite', expire_after=60)
    url = 'https://httpbingo.org/uuid'

    r1 = _retry(lambda: session.get(url))
    r2 = _retry(lambda: session.get(url))

    assert r1.from_cache is False
    assert r2.from_cache is True
    assert r1.text == r2.text  # same cached body
    assert db_path.exists()
    print(f"db size = {db_path.stat().st_size} bytes")
    print(f"cache backend = {type(session.cache).__name__}")


In [ ]:
# Example 3: install_cache + uninstall_cache patches requests globally.
import requests

import requests_cache

requests_cache.install_cache('demo_cache', backend='memory', expire_after=60)
try:
    r1 = _retry(lambda: requests.get('https://httpbingo.org/get'))
    r2 = _retry(lambda: requests.get('https://httpbingo.org/get'))
    assert getattr(r1, 'from_cache', False) is False
    assert getattr(r2, 'from_cache', False) is True
    print(f"patched: {requests_cache.is_installed()}")
finally:
    requests_cache.uninstall_cache()

assert requests_cache.is_installed() is False
print("uninstall ok")


In [ ]:
# Example 4: per-URL expiration patterns + cache key inspection.
from datetime import timedelta

from requests_cache import CachedSession

urls_expire_after = {
    'httpbingo.org/get': timedelta(seconds=30),
    'httpbingo.org/uuid': timedelta(seconds=5),
    '*.example.com': 60,
}
session = CachedSession(backend='memory', urls_expire_after=urls_expire_after)

r = _retry(lambda: session.get('https://httpbingo.org/get'))
keys = list(session.cache.responses.keys())
print(f"cached keys count = {len(keys)}")
print(f"first key (truncated) = {keys[0][:32]}...")

assert r.status_code == 200
assert len(keys) >= 1

In [ ]:
# Example 5: filter_fn to skip caching specific responses + cache clear/delete.
from requests_cache import CachedSession

def only_cache_200(response):
    return response.status_code == 200

session = CachedSession(
    backend='memory',
    expire_after=60,
    filter_fn=only_cache_200,
    allowable_codes=(200,),
)

r_ok = _retry(lambda: session.get('https://httpbingo.org/status/200'))
r_404 = _retry(lambda: session.get('https://httpbingo.org/status/404'))

r_ok_again = _retry(lambda: session.get('https://httpbingo.org/status/200'))
r_404_again = _retry(lambda: session.get('https://httpbingo.org/status/404'))

assert r_ok_again.from_cache is True
assert r_404_again.from_cache is False

session.cache.clear()
r_ok_after_clear = _retry(lambda: session.get('https://httpbingo.org/status/200'))
assert r_ok_after_clear.from_cache is False
print("filter_fn + clear ok")
